<a href="https://colab.research.google.com/github/SnehPhilip/GoogleEarthEngine/blob/main/gee_NDVIcalculation_Timeseries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vegetation Index calculation and Time series data preparation



#### This script calculates the Normalized Difference Vegetation Index (NDVI) for a specific point using Landsat 8 data and generates a time series chart in the console.


# 🛰️ Google Earth Engine Vegetation Dataset Catalog

This reference guide summarizes the primary datasets used for vegetation monitoring within Google Earth Engine (GEE).

---

### 1. Pre-Calculated Products (Ready-to-Use)
These datasets are **"plug-and-play."** The vegetation indices are already processed as individual bands, removing the need for manual algebraic calculations.

| Dataset Name | Snippet ID | Frequency | Resolution | Key Indices |
| :--- | :--- | :--- | :--- | :--- |
| **MODIS Terra Daily** | `MODIS/MOD09GA_006_NDVI` | Daily | 463m | NDVI |
| **Landsat 8-Day** | `LANDSAT/COMPOSITES/C02/T1_L2_8DAY_NDVI` | 8 Days | 30m | NDVI |
| **MODIS 16-Day Global** | `MODIS/061/MOD13Q1` | 16 Days | 250m | NDVI, EVI |

> **Note:** The **MOD13Q1**  includes the **Enhanced Vegetation Index (EVI)**, which corrects for atmospheric noise.


---

## 2. Raw Imagery (Manual Calculation)
Use these collections if requirement is for **high spatial resolution (10m)** or need the most recent available satellite pass. You must calculate the index manually using the NIR and Red bands.

| Source | Snippet ID | NIR Band | Red Band | Resolution |
| :--- | :--- | :--- | :--- | :--- |
| **Sentinel-2 (MSI)** | `COPERNICUS/S2_SR_HARMONIZED` | `B8` | `B4` | **10m** |
| **Landsat 8/9 (OLI)** | `LANDSAT/LC08/C02/T1_L2` | `SR_B5` | `SR_B4` | 30m |
| **Landsat 7 (ETM+)** | `LANDSAT/LE07/C02/T1_L2` | `SR_B4` | `SR_B3` | 30m |

[Image comparing spatial resolution of Sentinel-2 vs Landsat 8]

---

## 📐 Mathematical Implementation
When using the **Raw Imagery** table above, the Normalized Difference Vegetation Index (NDVI) is calculated using the following formula:

$$\text{NDVI} = \frac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red}}$$

  //addNDVI(image):
  
  ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
  image.addBands(ndvi)


### 1. Define a Region of interest (ROI) based on latitude /longitude

    var roi = ee.Geometry.Point([-122.2531, 37.6295]);

### 2. Load and filter the Landsat 8 collection based on required date
### 3. Calculate NDVI: (NIR - Red) / (NIR + Red)
    
    
    


    // Landsat 8 bands: B5 is NIR, B4 is Red
    
    var collection = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
      .filterBounds(roi)
      .filterDate('2022-01-01', '2023-12-31')
      .map(function(image) {
                var ndvi = image.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI');
        return image.addBands(ndvi);
      });

    // 4. Create the Time Series Chart
            var chart = ui.Chart.image.series({
              imageCollection: collection.select('NDVI'),
              region: roi,
              reducer: ee.Reducer.mean(),
              scale: 30
            }).setOptions({
              title: 'NDVI Time Series for Selected Location',
              vAxis: {title: 'NDVI Value'},
              hAxis: {title: 'Date'},
              lineWidth: 1,
              pointSize: 3,
              series: {0: {color: 'green'}}
            });

    // 5. Display the chart in the Console tab
          print(chart);



    Data Preparation:
    Use filterDate() and filterBounds() to narrow down your ImageCollection BY DATE AND latitude/longitude

    Index Calculation:
    Use .map() to apply a function (like NDVI or EVI) across every image in the collection.

    Charting:
    The ui.Chart.image.series() function is the standard tool for plotting a single location's data over time.

    Satellite records the intensity of light reflected off the surface separately.(splitting light into multiple "slices" or channels. Each slice is a band.)
        
    For Landsat 8
    Band 4	is Red	viz 0.64 – 0.67 μm on the em spectrum  
    (Distinguishing vegetation from soil.)
    
    Band 5	is Near-Infrared	viz 0.85 – 0.88 μm on the em spectrum
    (Critical for NDVI; measuring biomass.)
    

In [ ]:
import ee
import geemap

# 1. Initialize the Earth Engine library
# If running for the first time in Colab, uncomment the next line:
# ee.Authenticate()
ee.Initialize()

# 2. Define a point of interest (ROI)
roi = ee.Geometry.Point([-122.2531, 37.6295])

# 3. Define the NDVI function
def calculate_ndvi(image):
    # Landsat 8 bands: B5 is NIR, B4 is Red
    ndvi = image.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
    return image.addBands(ndvi)

# 4. Load and filter the Landsat 8 collection
collection = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
              .filterBounds(roi)
              .filterDate('2022-01-01', '2023-12-31')
              .map(calculate_ndvi))

# 5. Select the NDVI band for analysis
ndvi_col = collection.select('NDVI')

print("Collection processed. Number of images:", ndvi_col.size().getInfo())

In [ ]:
import geemap.chart as chart

# 4. Create the Time Series Chart
# Note: In Python/geemap, we specify the 'system:time_start' property for the x-axis
options = {
    'title': 'NDVI Time Series for Selected Location',
    'vAxis': {'title': 'NDVI Value'},
    'hAxis': {'title': 'Date'},
    'lineWidth': 1,
    'pointSize': 3,
    'series': {
        '0': {'color': 'green'}
    }
}

# Create the chart using geemap
ndvi_chart = chart.image_series(
    image_collection=collection.select('NDVI'),
    region=roi,
    reducer=ee.Reducer.mean(),
    scale=30,
    x_property='system:time_start'
)

# 5. Apply the options and display
# (Geemap charts are interactive by default in Colab)
ndvi_chart.set_options(**options)
ndvi_chart